In [ ]:
import os
import pandas as pd
import numpy as np

from src.synthetic_control_batch import benchmark_sweep

# Load specifications

In [ ]:
data_dir = os.path.join('..', '..', 'data')
syn_control_dir = os.path.join(data_dir, '2K500', 'synthetic_control')

specification_names = [
    'Text Persona - GPT4.1-mini',
    'JSON Persona - GPT4.1',
    'JSON Persona - GPT4.1-mini',
    'JSON Persona (Predicted Output) - GPT4.1',
    'JSON Persona (Predicted Output) - GPT4.1-mini',
    'Text Persona (Default Temperature) - GPT4.1-mini',
    'Text Persona (Reasoning) - GPT4.1-mini',
    'Text Persona (Repeating Questions) - GPT4.1-mini',
    'Text Persona - Gemini-Flash2.5',
    'LLM Finetuning (500 training samples) - GPT4.1-mini',
    'Demographics Only - GPT4.1-mini',
    'Persona Summary - GPT4.1-mini',
    'Persona Summary - JSON Persona - GPT4.1-mini',
]

specs = []
for name in specification_names:
    spec_dir = os.path.join(syn_control_dir, name)
    real_df = pd.read_csv(os.path.join(spec_dir, 'real.csv'), index_col=0)
    llm_df = pd.read_csv(os.path.join(spec_dir, 'LLM.csv'), index_col=0)
    specs.append({
        'name': name,
        'real': real_df.to_numpy(),
        'synthetic': llm_df.to_numpy(),
    })

print(f'Loaded {len(specs)} specifications')
for s in specs:
    print(f"  {s['name']}: {s['real'].shape}")

In [ ]:
output_dir = os.path.join('..', '..', 'outputs', 'synthetic_control', '2K500_batch')
os.makedirs(output_dir, exist_ok=True)

# Define methods

In [ ]:
# Best column-wise parameters
methods_column = [
    {'label': 'Ridge',                'method': 'ridge',                  'regularization_multiplier': 100},
    {'label': 'Lasso',                'method': 'lasso',                  'regularization_multiplier': 0.001},
    {'label': 'Elastic Net',          'method': 'elastic_net',            'regularization_multiplier': 0.01, 'en_l1_ratio': 0.3},
    {'label': 'Synthetic Control',    'method': 'synthetic_control',      'regularization_multiplier': 1e-6},
    {'label': 'Neural Net',           'method': 'neural_net',             'nn_hidden_dims': [8], 'nn_epochs': 200, 'nn_lr': 1e-3, 'nn_weight_decay': 5e-2, 'nn_batch_size': 128, 'nn_patience': 20, 'nn_seed': 42},
    {'label': 'MC Hard SVD',          'method': 'mc_hard_svd',            'mc_rank': 5},
    {'label': 'MC Soft SVD',          'method': 'mc_soft_svd',            'mc_rank': 20, 'mc_lambda': 20},
    {'label': 'MC ALS',               'method': 'mc_als',                 'mc_rank': 20, 'mc_lambda': 20},
    {'label': 'MC Synthetic Prior',   'method': 'mc_synthetic_prior',     'mc_rank': 8},
    {'label': 'Synthetic Intervention',  'method': 'synthetic_intervention', 'si_rank': 20, 'regularization_multiplier': 100},
]

# Best row-wise parameters
methods_row = [
    {'label': 'Ridge',                'method': 'ridge',                  'regularization_multiplier': 5000},
    {'label': 'Lasso',                'method': 'lasso',                  'regularization_multiplier': 1},
    {'label': 'Elastic Net',          'method': 'elastic_net',            'regularization_multiplier': 1, 'en_l1_ratio': 0.1},
    {'label': 'Synthetic Control',    'method': 'synthetic_control',      'regularization_multiplier': 1e-6},
    {'label': 'Neural Net',           'method': 'neural_net',             'nn_hidden_dims': [8, 8], 'nn_epochs': 200, 'nn_lr': 1e-3, 'nn_weight_decay': 1e-3, 'nn_batch_size': 128, 'nn_patience': 20, 'nn_seed': 42},
    {'label': 'MC Hard SVD',          'method': 'mc_hard_svd',            'mc_rank': 2},
    {'label': 'MC Soft SVD',          'method': 'mc_soft_svd',            'mc_rank': 10, 'mc_lambda': 10},
    {'label': 'MC ALS',               'method': 'mc_als',                 'mc_rank': 15, 'mc_lambda': 0.5},
    {'label': 'MC Synthetic Prior',   'method': 'mc_synthetic_prior',     'mc_rank': 2},
    {'label': 'Synthetic Intervention',  'method': 'synthetic_intervention', 'si_rank': 30, 'regularization_multiplier': 100},
]

# Run benchmark

In [ ]:
results_col = benchmark_sweep(
    specs=specs,
    methods=methods_column,
    direction='column',
    sc_kwargs={'imputation_rank': 5, 'min_col_std': 1},
    n_jobs=-2,
    verbose=False,
)

# Column-wise results

In [ ]:
results_col['formatted']

In [ ]:
results_col['correlation_mean']

# Save results

In [ ]:
results_col['correlation_mean'].to_csv(os.path.join(output_dir, 'column_corr_mean.csv'))
results_col['correlation_se'].to_csv(os.path.join(output_dir, 'column_corr_se.csv'))
results_col['formatted'].to_csv(os.path.join(output_dir, 'column_formatted.csv'))

# Row-wise benchmark

In [ ]:
specs_row = [s for s in specs if s['name'] == 'Text Persona - GPT4.1-mini']

results_row = benchmark_sweep(
    specs=specs_row,
    methods=methods_row,
    direction='row',
    sc_kwargs={'imputation_rank': 5, 'min_col_std': 1},
    n_jobs=-2,
    verbose=False
)

# Row-wise results

In [ ]:
results_row['formatted']

In [ ]:
results_row['correlation_mean']

# Save results

In [ ]:
results_row['correlation_mean'].to_csv(os.path.join(output_dir, 'row_corr_mean.csv'))
results_row['correlation_se'].to_csv(os.path.join(output_dir, 'row_corr_se.csv'))
results_row['formatted'].to_csv(os.path.join(output_dir, 'row_formatted.csv'))